# 07 - Article Type and Method Labels

                This notebook creates transparent rule-based prelabels for contribution type and methodological approach. It also exports a stratified validation sample for human review or later LLM-assisted labeling.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from hsr_analysis.common import *

import numpy as np
import pandas as pd

ensure_project_structure(ROOT)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)

print(f"Project root: {ROOT}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import re

articles = pd.read_csv(ROOT / "outputs/tables/articles_clean.csv")
article_topics = pd.read_csv(ROOT / "outputs/tables/article_topics.csv")
df = articles.merge(article_topics[["article_id", "topic_id", "topic_label_human", "topic_macro_category"]], on="article_id", how="left")
df["label_text"] = df[["title", "abstract_en", "abstract_de", "keywords_en", "type_en"]].fillna("").agg(" ".join, axis=1).map(lambda x: x.lower())

In [ ]:
def contribution_type(text, row):
    if bool(row.get("is_editorial_or_intro", False)):
        return "editorial_introduction"
    if bool(row.get("is_review", False)) or bool(row.get("is_bibliography", False)):
        return "review_or_bibliography"
    if re.search(r"\bdatabase\b|\bdata archive\b|\binfrastructure\b|\bdocumentation\b|\bdataset\b", text):
        return "research_infrastructure_or_data"
    if re.search(r"\bmethod\b|\bmethodology\b|\bmeasurement\b|\bcoding\b|\bsampling\b|\bclassification\b", text):
        return "methodological"
    if re.search(r"\bempirical\b|\banalysis\b|\bstudy\b|\bsurvey\b|\binterview\b|\bcase study\b|\bdata\b", text):
        return "empirical_analysis"
    if re.search(r"\btheory\b|\bconceptual\b|\bconcept\b|\bframework\b|\bcritique\b|\bnormative\b", text):
        return "conceptual_theoretical"
    return "mixed_or_unclear"

def method_approach(text):
    if re.search(r"\bregression\b|\bstatistic\b|\bquantitative\b|\bsurvey\b|\bpanel\b|\btime series\b|\bmodel\b", text):
        return "quantitative"
    if re.search(r"\binterview\b|\bethnograph\b|\bqualitative\b|\bgrounded theory\b|\bfocus group\b", text):
        return "qualitative"
    if re.search(r"\bmixed methods\b|\bmixed-method\b", text):
        return "mixed_methods"
    if re.search(r"\bdigital\b|\bcomputational\b|\balgorithm\b|\bgis\b|\bnetwork\b|\btext mining\b|\bmachine learning\b", text):
        return "computational_digital"
    if re.search(r"\barchive\b|\bhistorical narrative\b|\bsource\b|\bcase history\b", text):
        return "historical_narrative_or_archival"
    if re.search(r"\bformal\b|\bmathematical\b|\bgame theory\b", text):
        return "formal_theoretical"
    if re.search(r"\bconceptual\b|\bnormative\b|\btheoretical\b|\btheory\b", text):
        return "conceptual_normative"
    if re.search(r"\binfrastructure\b|\bdocumentation\b|\bdatabase\b|\bdata archive\b", text):
        return "infrastructure_documentation"
    return "unclear"

labels = df[["article_id", "year", "period", "language", "type_en", "topic_id", "topic_label_human", "topic_macro_category"]].copy()
labels["contribution_type_rule"] = df.apply(lambda row: contribution_type(row["label_text"], row), axis=1)
labels["method_approach_rule"] = df["label_text"].map(method_approach)
labels["contribution_type_human"] = ""
labels["method_approach_human"] = ""
labels["label_status"] = "rule_based_preset"
write_csv(labels, ROOT / "outputs/tables/article_type_labels.csv")

sample_size = max(100, int(0.05 * len(labels)))
sample_size = min(sample_size, len(labels))
strata = labels[["period", "language", "topic_macro_category", "contribution_type_rule"]].fillna("unknown").agg("|".join, axis=1)
try:
    if strata.value_counts().min() >= 2 and strata.nunique() < sample_size:
        sample, _ = train_test_split(labels, train_size=sample_size, random_state=42, stratify=strata)
    else:
        sample = labels.sample(n=sample_size, random_state=42)
except ValueError:
    sample = labels.sample(n=sample_size, random_state=42)
sample = sample.sort_values(["period", "topic_macro_category", "year"])
sample["validator_1_contribution_type"] = ""
sample["validator_1_method_approach"] = ""
sample["validator_1_notes"] = ""
write_csv(sample, ROOT / "outputs/labels/article_type_validation_sample.csv")

validation_path = ROOT / "outputs/labels/article_type_validation_sample.csv"
validated = pd.read_csv(validation_path)
results = []
if "validator_1_contribution_type" in validated.columns and validated["validator_1_contribution_type"].map(compact_ws).ne("").any():
    checked = validated[validated["validator_1_contribution_type"].map(compact_ws).ne("")]
    results.append({"metric": "n_validated_contribution_type", "value": len(checked)})
    results.append({"metric": "rule_human_agreement_contribution_type", "value": float((checked["validator_1_contribution_type"] == checked["contribution_type_rule"]).mean())})
if "validator_1_method_approach" in validated.columns and validated["validator_1_method_approach"].map(compact_ws).ne("").any():
    checked = validated[validated["validator_1_method_approach"].map(compact_ws).ne("")]
    results.append({"metric": "n_validated_method_approach", "value": len(checked)})
    results.append({"metric": "rule_human_agreement_method_approach", "value": float((checked["validator_1_method_approach"] == checked["method_approach_rule"]).mean())})
write_csv(pd.DataFrame(results, columns=["metric", "value"]), ROOT / "outputs/tables/article_type_validation_results.csv")
display(labels.head())

In [ ]:
type_year = labels.groupby(["year", "contribution_type_rule"]).size().reset_index(name="n")
type_year["share"] = type_year["n"] / type_year.groupby("year")["n"].transform("sum")
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=type_year, x="year", y="share", hue="contribution_type_rule", ax=ax)
ax.set_title("Contribution type over time")
ax.set_xlabel("Year")
ax.set_ylabel("Share")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), title="Contribution type")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_16_article_type_over_time.png", dpi=220)
save_caption(ROOT, "fig_16_article_type_over_time.png", "Rule-based contribution-type prelabels over time; final use requires validation.")
plt.show()

method_year = labels.groupby(["year", "method_approach_rule"]).size().reset_index(name="n")
method_year["share"] = method_year["n"] / method_year.groupby("year")["n"].transform("sum")
fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(data=method_year, x="year", y="share", hue="method_approach_rule", ax=ax)
ax.set_title("Methodological approach over time")
ax.set_xlabel("Year")
ax.set_ylabel("Share")
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5), title="Method")
fig.tight_layout()
fig.savefig(ROOT / "outputs/figures/fig_17_methodological_approach_over_time.png", dpi=220)
save_caption(ROOT, "fig_17_methodological_approach_over_time.png", "Rule-based method-approach prelabels over time; ambiguous records require human review.")
plt.show()